# NLP Sentiment Layer Analysis
## Does genuine news sentiment add predictive power beyond macro + market features?

**Setup:**
- Layer 3 replacement: GDELT historical Thailand financial news → VADER compound scores
- Optional upgrade: FinBERT re-scoring (run `python src/nlp_sentiment.py --finbert`)
- Targets: SET Index (1w/4w), Gold (1w/4w), USD/THB (1w/4w)
- Hypothesis (null): NLP sentiment does NOT improve out-of-sample DirAcc vs Macro+Market baseline

**Sections:**
1. Load & inspect raw GDELT data
2. Weekly feature aggregation
3. Lead-lag analysis (Gate: |r| ≥ 0.04 at k ≥ +1)
4. Regime-specific sentiment patterns
5. Ablation study: Model A (all layers) vs B (no NLP) vs C (macro only)
6. FinBERT vs VADER comparison
7. Recommendation

In [ ]:
import sys, warnings
warnings.filterwarnings("ignore")
sys.path.insert(0, "../../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy.stats import spearmanr
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import r2_score
import xgboost as xgb
import shap

from nlp_sentiment import (
    GDELTFetcher, FinBERTScorer, SentimentAggregator, LeadLagTester,
    run_ablation, GDELT_RAW, SENT_WEEKLY, PROC_DIR
)

%matplotlib inline
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})
print("All imports OK")

## 1. Raw GDELT Data

In [ ]:
raw = pd.read_csv(GDELT_RAW)
print(f"Shape: {raw.shape}")
print(f"Date range: {raw['published'].min()} → {raw['published'].max()}")
print(f"Columns: {list(raw.columns)}")
raw.head(3)

In [ ]:
# Coverage by year
raw["year"] = pd.to_datetime(raw["published"], errors="coerce").dt.year
coverage = raw.groupby("year").agg(
    n_articles=("title","count"),
    mean_compound=("weighted_compound","mean"),
    pct_positive=("sentiment", lambda x: (x=="positive").mean()),
    pct_negative=("sentiment", lambda x: (x=="negative").mean()),
).round(3)
print(coverage.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
coverage["n_articles"].plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set_title("Article Volume by Year"); axes[0].set_xlabel("")
coverage["mean_compound"].plot(kind="bar", ax=axes[1], color="coral")
axes[1].set_title("Mean Sentiment Score by Year"); axes[1].set_xlabel("")
axes[1].axhline(0, color="black", lw=0.8)
plt.tight_layout(); plt.show()

In [ ]:
# Sentiment distribution
fig, ax = plt.subplots(figsize=(8, 4))
raw["weighted_compound"].hist(bins=60, ax=ax, color="steelblue", alpha=0.75)
ax.axvline(0.05, color="green", lw=1.5, linestyle="--", label="positive threshold")
ax.axvline(-0.05, color="red", lw=1.5, linestyle="--", label="negative threshold")
ax.set_title("Distribution of Weighted VADER Compound Score")
ax.legend(); plt.tight_layout(); plt.show()

counts = raw["sentiment"].value_counts()
print("Sentiment class balance:")
for k, v in counts.items():
    print(f"  {k:10s}: {v:6d}  ({v/len(raw):.1%})")

## 2. Weekly Feature Aggregation
All features shifted 1 week to prevent look-ahead bias.

In [ ]:
use_finbert = "finbert_compound" in raw.columns
agg = SentimentAggregator()
sent_weekly = agg.aggregate(raw, use_finbert=use_finbert)
agg.save(sent_weekly)

print(f"Weekly rows: {len(sent_weekly)}")
print(f"Date range: {sent_weekly.index.min().date()} → {sent_weekly.index.max().date()}")
sent_weekly.describe().round(3)

In [ ]:
# Visualise key weekly features
market = pd.read_csv(PROC_DIR / "unified_weekly.csv", index_col=0, parse_dates=True)
common = sent_weekly.index.intersection(market.index)

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)
ax = axes[0]
ax.fill_between(common, sent_weekly.loc[common, "sent_mean_1w"],
                alpha=0.6, color="steelblue")
ax.axhline(0, color="black", lw=0.7)
ax.set_title("Weekly Sentiment (weighted VADER compound, 1-week lag)")

ax = axes[1]
market.loc[common, "SET_index_ret_w"].plot(ax=ax, color="darkgreen", lw=0.8)
ax.axhline(0, color="black", lw=0.7)
ax.set_title("SET Index Weekly Return")

ax = axes[2]
sent_weekly.loc[common, "news_vol_w"].plot(ax=ax, color="coral")
ax.set_title("News Article Volume (weekly count)")

ax = axes[3]
sent_weekly.loc[common, "sent_vol_4w"].plot(ax=ax, color="purple")
ax.set_title("Sentiment Volatility (4-week rolling std)")

plt.tight_layout(); plt.show()

## 3. Lead-Lag Analysis (Gate Test)
**Gate:** If no feature shows |Spearman r| ≥ 0.04 at k ≥ +1, stop here — NLP sentiment is not a leading signal.

k > 0 means sentiment this week predicts returns k weeks later (leading).
k < 0 means returns lead sentiment (reactive/lagging — useless for prediction).

In [ ]:
tester = LeadLagTester()
ll_df  = tester.run(sent_weekly, market)
ll_df.to_csv(PROC_DIR / "sentiment_leadlag.csv", index=False)
gate   = tester.print_summary(ll_df)
leading_features = [f for f, passed in gate.items() if passed]
print(f"\nLeading features (pass gate): {leading_features}")

In [ ]:
# Heatmap: Spearman r by (feature × lag) for SET target
import matplotlib.colors as mcolors

target = "SET_index_ret_w"
sub = ll_df[ll_df["target"] == target]
pivot = sub.pivot(index="feature", columns="lag", values="spearman_r")

fig, ax = plt.subplots(figsize=(10, max(4, len(pivot) * 0.5 + 1)))
im = ax.imshow(pivot.values, cmap="RdYlGn", vmin=-0.15, vmax=0.15, aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"k={k:+d}" for k in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        v = pivot.values[i, j]
        ax.text(j, i, f"{v:.2f}", ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.07 else "black")
ax.axvline(3.5, color="black", lw=2, label="k=0")  # k=0 column
ax.set_title(f"Lead-Lag Spearman r: Sentiment Features → {target}")
plt.colorbar(im, ax=ax, label="Spearman r")
plt.tight_layout(); plt.show()

## 4. Regime-Specific Sentiment Patterns

In [ ]:
# Assign regimes from VIX
vix = market["vix_price"].reindex(common)
p33, p67 = vix.quantile(0.33), vix.quantile(0.67)
regime = pd.cut(vix, bins=[-np.inf, p33, p67, np.inf],
                labels=["Bull", "Normal", "Crisis"])

sent_reg = sent_weekly.loc[common].copy()
sent_reg["regime"] = regime
sent_reg["SET_ret"] = market.loc[common, "SET_index_ret_w"]

print("Sentiment by regime:")
print(sent_reg.groupby("regime")[["sent_mean_1w","news_vol_w","sent_spike"]].mean().round(3))

# DirAcc of sentiment sign vs next-week SET return
sent_reg["sent_sign"] = np.sign(sent_reg["sent_mean_1w"])
sent_reg["correct"]   = (sent_reg["sent_sign"] == np.sign(sent_reg["SET_ret"])).astype(float)
print("\nDirectional accuracy of raw sentiment sign:")
print(sent_reg.groupby("regime")["correct"].mean().round(3))

In [ ]:
# Timeline: crisis periods highlighted
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(common, sent_weekly.loc[common, "sent_mean_1w"], color="steelblue",
        lw=0.8, alpha=0.8, label="Sent mean")
ax.fill_between(common,
                sent_weekly.loc[common, "sent_mean_1w"].where(regime == "Crisis"),
                alpha=0.4, color="red", label="Crisis period")
ax.fill_between(common,
                sent_weekly.loc[common, "sent_mean_1w"].where(regime == "Bull"),
                alpha=0.3, color="green", label="Bull period")
ax.axhline(0, color="black", lw=0.7)
ax.legend(loc="upper left")
ax.set_title("Sentiment Score with Regime Overlay")
plt.tight_layout(); plt.show()

## 5. Ablation Study
Compares three models:
- **Model A:** Macro + Market + NLP (full stack)
- **Model B:** Macro + Market (no NLP)
- **Model C:** Macro only (baseline)

Only features that passed the lead-lag gate are included in Model A.

In [ ]:
if not leading_features:
    print("No features passed the lead-lag gate.")
    print("Recommendation: NLP sentiment does not add predictive value at weekly horizon.")
    print("Options: (1) try 4w horizon, (2) use FinBERT, (3) remove Layer 3 entirely")
else:
    print(f"Running ablation with {len(leading_features)} leading features: {leading_features}")
    ablation_df = run_ablation(sent_weekly, market, leading_features, n_trials=20)
    ablation_df.to_csv(PROC_DIR / "sentiment_ablation.csv", index=False)
    print("\nAblation saved → sentiment_ablation.csv")

In [ ]:
# Display comparison table
if "ablation_df" in dir() and not ablation_df.empty:
    print("\n=== ABLATION RESULTS ===")
    print(f"{'Target':25s}  {'Model':30s}  {'DirAcc':>8}  {'R²':>8}  {'Sharpe':>8}")
    print("-" * 85)
    for _, r in ablation_df.sort_values(["target","model"]).iterrows():
        print(f"{r['target']:25s}  {r['model']:30s}  {r['dir_acc']:>8.1%}  "
              f"{r['r2']:>+8.4f}  {r['sharpe']:>+8.2f}")

    # Delta: A vs B (NLP contribution)
    print("\n=== NLP CONTRIBUTION (Model A − Model B) ===")
    for tgt in ablation_df["target"].unique():
        sub = ablation_df[ablation_df["target"] == tgt]
        a = sub[sub["model"].str.startswith("A")].iloc[0] if any(sub["model"].str.startswith("A")) else None
        b = sub[sub["model"].str.startswith("B")].iloc[0] if any(sub["model"].str.startswith("B")) else None
        if a is not None and b is not None:
            dda = a["dir_acc"] - b["dir_acc"]
            dr2 = a["r2"]      - b["r2"]
            dsh = a["sharpe"]  - b["sharpe"]
            sym = "↑ HELPS" if dda > 0.005 else ("↓ HURTS" if dda < -0.005 else "→ neutral")
            print(f"  {tgt:25s}  ΔDirAcc={dda:+.1%}  ΔR²={dr2:+.4f}  ΔSharpe={dsh:+.2f}  {sym}")

## 6. FinBERT vs VADER Comparison
FinBERT is a domain-specific financial BERT model. It should classify financial text
more accurately than general-purpose VADER, especially for nuanced sentences.

To run FinBERT: `python src/nlp_sentiment.py --finbert`
(CPU: ~0.5s/article — run overnight on full dataset)
(GPU: ~0.02s/article — practical in ~30 min)

In [ ]:
if "finbert_compound" in raw.columns:
    # Compare VADER vs FinBERT agreement
    agree = np.sign(raw["compound"]) == np.sign(raw["finbert_compound"])
    disagree_pct = 1 - agree.mean()
    print(f"VADER vs FinBERT: {disagree_pct:.1%} disagreement rate")

    # Correlation
    from scipy.stats import pearsonr
    r, p = pearsonr(raw["compound"].fillna(0), raw["finbert_compound"].fillna(0))
    print(f"Pearson r = {r:.3f}  (p={p:.3e})")

    # Compare weekly aggregates
    raw2 = raw.copy()
    raw2["weighted_compound"] = raw2["finbert_compound"]  # swap scorer
    agg2 = SentimentAggregator()
    fb_weekly = agg2.aggregate(raw2, use_finbert=False)
    fb_weekly.columns = [f"fb_{c}" for c in fb_weekly.columns]

    # Lead-lag with FinBERT
    fb_cols = [c for c in fb_weekly.columns if "sent_mean" in c]
    ll_fb = tester.run(fb_weekly.rename(columns={c: c.replace("fb_","") for c in fb_cols}), market)
    gate_fb = tester.print_summary(ll_fb)
    print("\nFinBERT leading features:", [f for f, p in gate_fb.items() if p])
else:
    print("FinBERT scores not available. Run: python src/nlp_sentiment.py --finbert")
    print("Requires transformers package + patience (CPU: ~14 hours for 100k articles)")

## 7. Recommendation

In [ ]:
print("=" * 70)
print("RECOMMENDATION LOGIC")
print("=" * 70)

passed_gate = len(leading_features) > 0

if "ablation_df" in dir() and not ablation_df.empty:
    # Check if Model A consistently beats Model B
    ab_compare = ablation_df.groupby("model").agg(
        mean_da=("dir_acc","mean"), mean_r2=("r2","mean"), mean_sh=("sharpe","mean")
    )
    model_a = ab_compare.filter(like="A:", axis=0)
    model_b = ab_compare.filter(like="B:", axis=0)
    if not model_a.empty and not model_b.empty:
        da_delta = float(model_a["mean_da"]) - float(model_b["mean_da"])
        sh_delta = float(model_a["mean_sh"]) - float(model_b["mean_sh"])
        verdict_nlp = (da_delta > 0.005 or sh_delta > 0.1)
    else:
        verdict_nlp = False
else:
    verdict_nlp = False

print()
if not passed_gate:
    print("VERDICT: REMOVE NLP Layer 3")
    print("  Lead-lag gate FAILED: no sentiment feature shows |r| ≥ 0.04 at k ≥ +1")
    print("  Sentiment appears to be coincident or lagging, not predictive")
    print()
    print("  Recommended path:")
    print("  1. Try FinBERT re-scoring (--finbert) — more accurate classification")
    print("  2. Try 4-week horizon — sentiment may predict medium-term better")
    print("  3. Explore entity-level sentiment (BOT-specific, Fed-specific)")
    print("  4. If still no signal: accept that weekly NLP is not informative for")
    print("     Thai market at this granularity")
elif not verdict_nlp:
    print("VERDICT: MARGINAL — NLP passes gate but does not improve ablation")
    print("  Sentiment LEADS at some lag but XGBoost does not use it effectively")
    print()
    print("  Recommended path:")
    print("  1. Add FinBERT compound as a direct feature (alongside VADER)")
    print("  2. Try sentiment × regime interactions:")
    print("     e.g. sent_mean_1w × is_crisis (crisis amplifies news impact)")
    print("  3. Use FinBERT 4-class model: very positive / positive / negative / very negative")
    print("  4. Consider topic-specific sentiment (policy vs market vs macro)")
else:
    print("VERDICT: KEEP NLP Layer 3")
    print(f"  {len(leading_features)} features pass lead-lag gate AND ablation shows improvement")
    print(f"  Leading features to include: {leading_features}")
    print()
    print("  Next steps:")
    print("  1. Run --finbert to upgrade from VADER to FinBERT scores")
    print("  2. Add leading features to notebook 03 feature engineering")
    print("  3. Set MIN_CORR=0.04 gate in feature-catalog cell")
    print("  4. Re-run full model and compare walk-forward metrics")

print()
print("=" * 70)